Data wrangling and filtering for private landowner analysis

In [1]:
import pandas as pd

stats = pd.read_excel("ParcelOutputs/parcel_filter_1a.xlsx")
tia_parcels = pd.read_excel("ParcelOutputs/parcel_filter_1b_edit.xlsx")

tia_parcels = tia_parcels.merge(stats, how="left", on=["TIA_ID", "owner"])

tia_parcels.to_excel("ParcelOutputs/parcel_filter_1c.xlsx", index=False)



In [11]:
import pandas as pd
tia_parcels = pd.read_excel("ParcelOutputs/parcel_filter_1b_edit.xlsx")

tia_parcels["area_acres_geo"] = pd.to_numeric(
    tia_parcels["area_acres_geo"], errors="coerce"
)

tia_parcels["SUM_area_acres_geo"] = tia_parcels.groupby(["TIA_ID", "owner"])["area_acres_geo"].transform("sum")
tia_parcels["Acres_summed"] = tia_parcels.groupby(["TIA_ID"])["area_acres_geo"].transform("sum")

tia_parcels["Acres"] = pd.to_numeric(
    tia_parcels["Acres"], errors="coerce"
)

tia_parcels["tia_owner_pct"] = (tia_parcels["SUM_area_acres_geo"] / tia_parcels["Acres_summed"])*100

tia_parcels_filtered = tia_parcels[(tia_parcels["SUM_area_acres_geo"]>600) | (tia_parcels["tia_owner_pct"]>10)]

cols = list(tia_parcels_filtered.columns[:26])
cols.extend(['owner', 'parcelnumb', 'area_acres_geo', 'SUM_area_acres_geo', 'tia_owner_pct'])
tia_parcels_filtered = tia_parcels_filtered[cols]

tia_parcels_filtered.to_excel("ParcelOutputs/tia_parcels_filtered.xlsx", index=False)


In [14]:
summary = (
    tia_parcels_filtered
    .groupby("owner")
    .agg(
        num_TIAs=("TIA_ID", "nunique"),
        num_high_priority_TIAs=(
            "TIA_ID",
            lambda x: x[
                tia_parcels_filtered.loc[x.index, "BiologicalPriority"] == 1
            ].nunique()
        ),
        tia_numbers=("TIA_ID", lambda x: list(pd.unique(x))),
        parcel_numbers=("parcelnumb", lambda x: list(pd.unique(x))),
        num_Disciplines=("Discipline", "nunique"),
        created_by=("CreatedBy", lambda x: list(pd.unique(x))),
        counties=("CountyName", lambda x: list(pd.unique(x))),
        max_comp_pct=("tia_owner_pct", "max"),
        max_comp_acres=("SUM_area_acres_geo", "max"), # the maximum acreage within a single TIA owned by that landowner
        total_comp_acres=("area_acres_geo", "sum")
    )
    .reset_index()
)

summary["max_comp_pct"] = summary["max_comp_pct"].round(2)
summary["max_comp_acres"] = summary["max_comp_acres"].round(2)
summary["total_comp_acres"] = summary["total_comp_acres"].round(2)

summary.to_excel("ParcelOutputs/summary.xlsx", index=False)

PermissionError: [Errno 13] Permission denied: 'ParcelOutputs/summary.xlsx'

In [ ]:
import re

non_private_owners = [
  "UNITED STATES OF AMERICA", "COLORADO, STATE OF", "COLORADO DIVISION OF WILDLIFE", 
  "UNITED STATES OF AMERICA  BLM", "STATE OF COLORADO", "USFS", "BLM", 
  "UNITED STATES OF AMERICA RIO GRANDE NATIONAL FOREST", "BUREAU OF LAND MANAGEMENT", 
  "UNITED STATES OF AMERICA RIO GRANDE NATIONAL FOREST, WASHINGTON DC", 
  "U.S. FOREST SERVICE", "UNITED STATES OF AMERICA, RIO GRANDE NATIONAL FOREST", 
  "FOREST SERVICE", "BUREAU OF LAND MANAGEMENT, U.S. DEPARTMENT OF INTERIOR", 
  "COUNTY OF LARIMER", "SAN JUAN NATIONAL FOREST", "UNITED STATES OF AMERICA  IN", 
  "COUNTY OF JEFFERSON", "STATE OF COLORADO DIVISION OF PARKS &", 
  "WHITE RIVER NATIONAL FOREST", "UNITED STATES GOVERNMENT", 
  "UNITED STATES OF AMERICA HQ BLDG", "U S POSTAL SERVICE", 
  "UNITED STATES OF AMERICA FOREST SERV; DEPT OF AGRICULTURE", 
  "U S GOVERNMENT", "US FOREST SERVICE", "CITY & COUNTY OF DENVER", 
  "DENVER MOUNTAIN PARKS", "SAN  JUAN NATIONAL FOREST", "COLORADO SPRINGS CITY OF", 
  "AIR FORCE ACADEMY", "UNITED STATES DEPT OF AG", "USDA FOREST SERVICE", 
  "U S GOVERNMENT/ROCKY MOUNTAIN NTNL PK", "CITY AND COUNTY OF BROOMFIELD", 
  "COLORADO PARKS AND WILDLIFE", "B.L.M.", "STATE OF COLORADO, DIV OF WILDLIFE FOR B",
  "NATIONAL PARK SERVICE", "COLORADO NATIONAL MONUMENT", "San Juan National Forest", 
  "Mesa Verde National Park", "Bureau of Land Management", "Bureau of Reclamation/USFS",
  "DIVISION OF WILDLIFE AND", "USFS PIKE", "UNITED STATES FOREST SERVICE", 
  "DIVISION OF WILDLIFE & WILDLIFE COMMISSION", "U S A",
  "STATE OF COLORADO STATE BOARD OF LAND COMMISSIONERS", "STATE  OF COLORADO", 
  "STATE BOARD OF LAND COMMISSIONERS", "UTE MOUNTAIN TRIBE", "BUREAU OF RECLAMATION",
  "U S FOREST SERVICE", "GRAND MESA NATIONAL FOREST FRUITA DIVISION", 
  "BROWN'S PARK NATIONAL WILDLIFE REFUGE", "Ute Mountain Ute Indian Reservation",
  "Canyons of the Ancients Monument", "Ute Mountain Ute Indian Tribal Park",
  "US GOVERNMENT", "STATE OF COLO DEPT OF NATURAL RESOURCES", "UTE MOUNTAIN UTE TRIBE", 
  "COLORADO STATE OF", "STATE OF COLORADO, BOARD OF LAND COMMISSIONERS",
  "STATE OF COLORADO, STATE BOARD OF LAND COMMISSIONERS", "Comanche National Grassland",
  "STATE LAND", "State", "STATE BOARD OF LAND COMMISSION", "STOUT ROBERT H & LETTA L",
"U.S. DEPARTMENT OF AGRICULTURE",
"PINION CANYON MANEUVER SITE", "U.S. DEPARTMENT OF THE INTERIOR"
]

more_public = ["Forest Service", "U S A", "STATE OF COLORADO", "COLORADO STATE OF", "CITY OF", "COUNTY OF", "UNITED STATES", "NATIONAL FOREST", "NATIONAL FORREST"]

# filter out public lands
summary = summary[~summary["owner"].isin(non_private_owners)]
summary = summary[~summary["owner"].isin(["PRIVATE", "Private"])]

pattern = "|".join(map(re.escape, more_public))

filtered = summary[
    ~summary["owner"].str.contains(pattern, case=False, na=False)
]

filtered.to_excel("ParcelOutputs/summary_private.xlsx", index=False)